<h1 style="text-align: center;">로봇 자율주행:<br>고수준 판단 및 저수준 제어</h1>

<br><br>

<div style="text-align: right; color: gray; font-style: italic;">
김규래&emsp;<br>
kkr.kyurae.kim@gmail.com&emsp;
</div><br>

---
---

## 0. 비상용

```bash
pkill -9 gz
pkill -9 gz_sim
pkill -9 ruby
```

---

## 1. 작업 폴더 생성

```bash
mkdir -p ~/robot_navigation/{colcon_ws/src,firmware,scripts}
```

---

## 2. Serial Port 권한 확인

```bash
groups
```

```text
>> ... dialout ... ...
```

---

## 3. Serial Port 권한 부여

```bash
sudo usermod -a -G dialout $USER
```

이후

```bash
reboot
```

#### 재부팅 후에 Serial Port 권한 재확인:
```bash
groups
```
```text
>> ... dialout ... ...
```
```bash
ls -l /dev/ttyUSB*
```
```text
>> crw-rw---- 1 root dialout ... ... /dev/ttyUSB0
```

---
---

## A. Git

**git 최초 실행:**

```bash
git config --global user.email "you@example.com"
git config --global user.name "Your Name"
```
<br>

**로그인 정보 저장:**
```bash
git config --global credential.helper store
```

---

## B. ROS2 Jazzy 설치

```bash
sudo apt install software-properties-common
sudo add-apt-repository universe
```

```bash
sudo apt update
sudo apt install curl -y
```

```bash
export ROS_APT_SOURCE_VERSION=$(curl -s https://api.github.com/repos/ros-infrastructure/ros-apt-source/releases/latest | grep -F "tag_name" | awk -F'"' '{print $4}')
```

```bash
curl -L -o /tmp/ros2-apt-source.deb "https://github.com/ros-infrastructure/ros-apt-source/releases/download/${ROS_APT_SOURCE_VERSION}/ros2-apt-source_${ROS_APT_SOURCE_VERSION}.$(. /etc/os-release && echo ${UBUNTU_CODENAME:-${VERSION_CODENAME}})_all.deb"
```

```bash
sudo dpkg -i /tmp/ros2-apt-source.deb
```

```bash
sudo apt update
sudo apt upgrade
```

```bash
sudo apt install ros-dev-tools
sudo apt install ros-jazzy-desktop
```

---

## C. VS Code 설치

```bash
cd ~/Downloads/
wget -O vscode.deb "https://update.code.visualstudio.com/latest/linux-deb-arm64/stable"
sudo apt install ./vscode.deb
```

---

## D. Arduino IDE 설치 및 커스텀 명령어 생성

#### 설치:

```bash
sudo apt update
sudo apt install -y curl libfuse2t64
```

```bash
sudo mkdir -p /opt/arduino-ide
cd /opt/arduino-ide
```

```bash
sudo curl -fL \
    https://github.com/arduino/arduino-ide/releases/latest/download/arduino-ide_2.3.10_Linux_64bit.AppImage \
    -o arduino-ide.AppImage
```

```bash
cd /opt/arduino-ide
sudo chmod +x arduino-ide.AppImage
```

#### 테스트 실행:

```bash
cd /opt/arduino-ide
./arduino-ide.AppImage --no-sandbox
```

#### 명령어 생성:

```bash
nano ~/robot_navigation/scripts/arduino-ide
```

```bash
#!/bin/sh
exec /opt/arduino-ide/arduino-ide.AppImage --no-sandbox "$@"
```

```bash
chmod +x ~/robot_navigation/scripts/arduino-ide
sudo ln -s $HOME/robot_navigation/scripts/arduino-ide /usr/local/bin/arduino-ide
```

---

## E. Groot2 설치 및 커스텀 명령어 생성

#### 설치:

```bash
sudo mkdir -p /opt/groot2
cd /opt/groot2
sudo wget -O groot2.AppImage https://pub-32cef6782a9e411e82222dee82af193e.r2.dev/Groot2-v1.10.0-x86_64.AppImage
```

#### 테스트 실행:

```bash
cd /opt/groot2
./groot2.AppImage
```

#### 명령어 생성:

```bash
nano ~/robot_navigation/scripts/groot2
```

```bash
#!/bin/sh
exec /opt/groot2/groot2.AppImage "$@"
```

```bash
chmod +x ~/robot_navigation/scripts/groot2
sudo ln -s $HOME/robot_navigation/scripts/groot2 /usr/local/bin/groot2
```

---

## F. 인터넷 브라우저 삭제

```bash
sudo snap remove --purge firefox 2>/dev/null || true
sudo snap remove --purge chromium 2>/dev/null || true
sudo apt purge -y firefox firefox-esr chromium chromium-browser google-chrome-stable
flatpak uninstall --system --delete-data -y org.mozilla.firefox 2>/dev/null || true
flatpak uninstall --system --delete-data -y org.chromium.Chromium 2>/dev/null || true
flatpak uninstall --system --delete-data -y com.google.Chrome 2>/dev/null || true
flatpak uninstall --user --delete-data -y org.mozilla.firefox 2>/dev/null || true
flatpak uninstall --user --delete-data -y org.chromium.Chromium 2>/dev/null || true
flatpak uninstall --user --delete-data -y com.google.Chrome 2>/dev/null || true
```

---

## G. PCB–ESP32 GPIO 구성

```cpp
// Motor
#define FR_IN1 26
#define FR_IN2 25

#define FL_IN1 19
#define FL_IN2 27

#define RR_IN1 23
#define RR_IN2 12

#define RL_IN1 0
#define RL_IN2 15

// IMU (MPU6050)
#define IMU_SDA 21
#define IMU_SCL 22

// LiDAR
#define LIDAR_RX 16

// Potentiometer
#define POT_PIN 39

// Servo
#define SERVO_PIN 14

// Buzzer
#define BUZZER_PIN 5
```

---

## H. Linux CLI

#### $1)$ 터미널 프롬프트 (prompt) 이해

터미널을 실행하면 보이는 문자열:

```bash
user_name@host_name:~/parent_directory/current_directory$
```

* user_name: 현재 로그인한 사용자 이름
* host_name: 현재 접속한 컴퓨터(호스트) 이름
* @: 사용자와 컴퓨터를 구분하는 표시
* ~: 사용자의 home 디렉토리(폴더)
* /: 폴더와 폴더를 구분하는 표시
* parent_directory: 상위(부모) 디렉토리
* current_directory: 현재 디렉토리
* $: 일반 사용자 권한을 나타내는 표시 (#: 관리자 권한)

#### $2)$ 기본 터미널 명령어 종류

(a) `cd`: 디렉토리 이동 (change directory)

```bash
cd target_directory
```

<br>(b) `cd`: 홈 디렉토리로 이동

```bash
cd
```

<br>(c) `cd ..`: 상위 디렉토리로 이동

```bash
cd ..
```

<br>(d) `pwd`: 현재 디렉토리 확인 (print working directory)

```bash
pwd
```

<br>(e) `ls`: 현재 디렉토리 내 파일 목록 확인 (list)

```bash
ls
```

<br>(f) `ls -a`: 현재 디렉토리 내 숨김 파일 포함 모든 파일 확인 (list -all)

```bash
ls -a
```

<br>(g) `mkdir`: 현재 경로에 새로운 폴더 생성 (make directory)

```bash
mkdir folder_name
```

<br>(h) `mkdir -p`: 현재 경로에 새로운 폴더 생성하며 상위 폴더까지 생성 (make directory -parent)

```bash
mkdir -p parent_folder_name/folder_name
```

<br>(i) `touch`: 현재 경로에 새로운 파일 생성

```bash
touch file_name.txt
```

<br>(j) `cp`: 파일 복사 (copy)

```bash
cp file_path/file_name copy_path/new_file_name
```

<br>(k) `mv`: 파일 이동 (move)

```bash
mv file_path/file_name move_path/new_file_name
```

<br>(l) `rm`: 파일 삭제 (remove)

```bash
rm file_path/file_name
```

<br>(m) `rm -r`: 해당 폴더 및 모든 내부 파일 삭제 (remove -recursive)

```bash
rm -r folder_path
```

<br>(n) rmdir: 빈 폴더 삭제 (remove directory)

```bash
rmdir empty_folder
```

<br>(o) echo: 입력한 문자열이나 값을 터미널에 출력 혹은 파일에 저장

```bash
echo hi
echo "hi"
echo $USER
echo "hi" > test.txt
echo "hello" > test.txt
echo "again" >> test.txt
```

<br>(p) chmod: 파일/폴더의 권한 변경 (change mode)

```bash
chmod +x file_name
chmod -x file_name
chmod +w file_name
chmod -w file_name
chmod 444 file_name
chmod 644 file_name
```

<br>(q) nano: 터미널에서 GUI없이 사용하는 텍스트 편집기

```bash
nano file_name
```

<br>(r) ping: 특정 컴퓨터(호스트)와 네트워크 연결 상태를 확인

```bash
ping 8.8.8.8
ping <ip_address>
ping <host_name>.local
```

IP 주소 확인법:

```bash
hostname -I
```

<br>(s) ssh: 원격 컴퓨터(Jetson)에 접속

```bash
ssh <user_name>@<host_name>
ssh <user_name>@<ip_address>
```

<br>(t) sudo: 관리자(root) 권한으로 명령어 실행

```bash
sudo 명령어
sudo rm 관리자_권한_폴더/관리자_권한_파일
sudo nano 관리자_권한_폴더/관리자_권한_파일
```

가장 많이 사용되는 `sudo` 명령어:<br>
`sudo apt update`: 패키지 목록 갱신<br>
`sudo apt upgrade`: 설치된 패키지를 최신 버전으로 업데이트<br>
`sudo apt install <package_name>`: 패키지 설치

---

## I. 유용한 ROS2 Commands

```bash
ros2 run [pkg] [executable]
ros2 run [pkg] [executable] --ros-args -p [param]:=[value]

ros2 node list
ros2 node info /node

ros2 topic list
ros2 topic list -t
ros2 topic list -v
ros2 topic echo /topic
ros2 topic info /topic
ros2 topic info /topic -v
ros2 topic pub /topic [msg_type] "[data]"
ros2 topic hz
ros2 topic find [msg_type]

ros2 service list
ros2 service list -t
ros2 service type /service
ros2 service find [srv_type]
ros2 service call /service [srv_type] "[data]"

ros2 action list
ros2 action list -t
ros2 action info /action
ros2 action info /action -t
ros2 action send_goal /action [action_type] "[data]"
ros2 action send_goal /action [action_type] "[data]" --feedback

ros2 param list
ros2 param get /node [param]
ros2 param set /node [param] [value]
ros2 param describe /node [param]

ros2 interface list
ros2 interface show [interface]
ros2 interface packages

ros2 doctor
ros2 doctor --report
ros2 wtf

rqt_graph
ros2 run tf2_tools view_frames
rviz2 -d [PATH]/[CONFIG_FILE]
```

---
---

<br><br><div style="text-align: right; color: gray; font-style: italic;">
© 2026 김규래 (Kyu Rae Kim). All rights reserved.&emsp;<br><br>
This material is provided solely for the intended instructional purpose.&emsp;<br>
Redistribution, reproduction, modification, adaptation, or reuse of this material in any form without prior written permission from the copyright holder is prohibited.&emsp;
</div>